# 10_very_high_sales_pipeline

Generated from `alteryx-workflows\10_very_high_sales_pipeline.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `raw_orders`, `customer_dim`, `targets`.


In [ ]:
raw_orders = spark.createDataFrame([["C001", "West", "Online", "2026-01-12", "SKU-A-100", "1200", "4"], ["C002", "East", "Store", "2026-01-18", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "2026-02-03", "SKU-A-140", "640", "1"], ["C003", "North", "Partner", "2026-02-11", "SKU-C-010", "2100", "6"], ["C004", "South", "Store", "2026-03-01", "SKU-B-220", "430", "3"], ["C002", "East", "Online", "2026-03-09", "SKU-A-100", "990", "2"], ["C005", "West", "Partner", "2026-03-21", "SKU-C-010", "150", "1"], ["C003", "North", "Store", "2026-04-02", "SKU-A-140", "1760", "5"]], ["CustomerID", "Region", "Channel", "OrderDate", "SKU", "Amount", "Units"])
customer_dim = spark.createDataFrame([["C001", "Acme Retail", "Gold"], ["C002", "Northwind", "Silver"], ["C003", "Contoso", "Gold"], ["C004", "Fabrikam", "Bronze"], ["C005", "Adventure", "Silver"]], ["CustomerID", "CustomerName", "Tier"])
targets = spark.createDataFrame([["West", "5000"], ["East", "3000"], ["North", "4000"], ["South", "2000"]], ["Region", "TargetAmount"])


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 3

Produces `select_raw`, `sku_parts_1`, `sku_parts`, `order_date`, and 4 more.


In [ ]:
select_raw = raw_orders
sku_parts_1 = raw_orders.withColumn("SKUFamily", F.regexp_extract(F.col("SKU"), "(?i)([A-Z]+)-([0-9]+)", 1))
sku_parts = sku_parts_1.withColumn("SKUNumber", F.regexp_extract(F.col("SKU"), "(?i)([A-Z]+)-([0-9]+)", 2))
order_date = sku_parts.withColumn("OrderDateParsed", F.to_date(F.col("OrderDate"), "yyyy-MM-dd"))
amount = order_date.withColumn("AmountNum", F.col("Amount").cast('double'))
units = amount.withColumn("UnitsNum", F.col("Units").cast('double'))
quality_flag = units.withColumn("QualityFlag", F.when(F.col("AmountNum").isNull() | (F.col("AmountNum") < 0), F.lit("Invalid")).otherwise(F.lit("Valid")))
valid_rows = quality_flag.filter(F.col("QualityFlag") == "Valid")


### Transformations 2 of 3

Produces `dim_unique`, `fact_to_dim`, `loyalty`, `time_order`, and 4 more.


In [ ]:
dim_unique = customer_dim.dropDuplicates(["CustomerID"])
fact_to_dim = valid_rows.join(dim_unique, (valid_rows["CustomerID"] == dim_unique["CustomerID"]), 'inner').drop(dim_unique["CustomerID"])
loyalty = fact_to_dim.withColumn("LoyaltyCredit", F.when(F.col("Tier") == "Gold", F.col("AmountNum") * 0.05).otherwise(F.when(F.col("Tier") == "Silver", F.col("AmountNum") * 0.02).otherwise(F.lit(0))))
time_order = loyalty.orderBy(F.col("OrderDate").asc())
running_amount = time_order.withColumn("RunningTotal", F.sum(F.col("Amount").cast('double')).over(Window.partitionBy(*["Region"]).orderBy(F.col("OrderDate").asc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)))
by_region = running_amount.groupBy(*["Region"]).agg(F.sum(F.col("AmountNum").cast('double')).alias("Sum_AmountNum"), F.count(F.col("AmountNum")).alias("RowCount"))
by_channel = running_amount.groupBy(*["Channel"]).agg(F.sum(F.col("AmountNum").cast('double')).alias("Sum_AmountNum"), F.count(F.col("AmountNum")).alias("RowCount"))
by_sku_family = running_amount.groupBy(*["SKUFamily"]).agg(F.sum(F.col("UnitsNum").cast('double')).alias("Sum_UnitsNum"), F.count(F.col("UnitsNum")).alias("RowCount"))


### Transformations 3 of 3

Produces `target`, `region_vs_target`, `gap`, `goal_status`, and 4 more.


In [ ]:
target = targets.withColumn("TargetAmount", F.col("TargetAmount").cast('double'))
region_vs_target = by_region.join(target, (by_region["Region"] == target["Region"]), 'inner').drop(target["Region"])
gap = region_vs_target.withColumn("Gap", F.col("Sum_AmountNum") - F.col("TargetAmount"))
goal_status = gap.withColumn("GoalStatus", F.when(F.col("Gap") >= 0, F.lit("Met")).otherwise(F.lit("Missed")))
channel_matrix = by_channel.groupBy(*["Region"]).pivot("Channel").agg(F.sum(F.col("Amount").cast('double')))
publish_stack = by_region.unionByName(by_channel, allowMissingColumns=True)
worst_gap = goal_status.orderBy(F.col("Gap").desc())
top_gaps = worst_gap.limit(10)


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(valid_rows)
display(publish_stack)
display(top_gaps)
display(goal_status)
display(channel_matrix)
display(by_sku_family)
display(running_amount)
